# Colab bootstrap

Run the cell below once after connecting to a new Colab runtime. It mounts Google Drive, clones or updates the repository, installs the project with the annotation-app dependencies, and configures the shared runtime paths.

In [1]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/terriljoel/retrieval-grounded-remote-sensing.git"
REPO_REF = "feat/object-detection-YOLO"  # Change to main after merging.
REPO_DIR = Path("/content/retrieval-grounded-remote-sensing")
SHARED_ROOT = Path("/content/drive/Othercomputers/My laptop/shared_resources")

drive.mount("/content/drive")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Repository path exists but is not a Git clone: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[annotation]"],
    check=True,
)

if not SHARED_ROOT.is_dir():
    raise FileNotFoundError(
        f"Shared resources were not found at {SHARED_ROOT}. "
        "Check the Google Drive computer and folder names."
    )

os.environ.update({
    "SHARED_RESOURCES_ROOT": str(SHARED_ROOT),
    "RAW_DATASET_ROOT": "/content/datasets/raw",
    "PROCESSED_DATASET_ROOT": "/content/datasets/processed",
    "MANIFEST_ROOT": str(SHARED_ROOT / "datasets" / "manifests"),
    "EXPERIMENT_OUTPUT_ROOT": str(SHARED_ROOT / "experiment_outputs"),
    "JOB_LOG_ROOT": str(SHARED_ROOT / "experiment_outputs" / "job_logs"),
    "INFERENCE_EXPORT_ROOT": str(SHARED_ROOT / "inference" / "object_detection_inference_new_remote_sensing_dataset_external-4"),
})

Path(os.environ["MANIFEST_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["EXPERIMENT_OUTPUT_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["JOB_LOG_ROOT"]).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

print(f"Ready. Working directory: {Path.cwd()}")
print("The detector commands and annotation app are ready.")

Mounted at /content/drive
Ready. Working directory: /content/retrieval-grounded-remote-sensing
The detector commands and annotation app are ready.


In [2]:
from getpass import getpass
import os

nim_api_key = os.environ.get("NIM_API_KEY", "").strip()

if not nim_api_key:
    nim_api_key = getpass("NVIDIA NIM API key: ").strip()

if not nim_api_key.startswith("nvapi-"):
    raise ValueError("The supplied key is not a valid NVIDIA NIM API key")

os.environ["NIM_API_KEY"] = nim_api_key

print("NVIDIA NIM API key configured for this runtime.")

NVIDIA NIM API key configured for this runtime.


In [14]:
%cd /content/retrieval-grounded-remote-sensing

!git pull --ff-only
!pip install -e ".[annotation]"

/content
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 20 (delta 14), reused 20 (delta 14), pack-reused 0 (from 0)
Unpacking objects: 100% (20/20), 3.78 KiB | 645.00 KiB/s, done.
From https://github.com/terriljoel/retrieval-grounded-remote-sensing
   c2827c0..efdbd35  feat/object-detection-YOLO -> origin/feat/object-detection-YOLO
Updating c2827c0..efdbd35
Fast-forward
 configs/evaluation/background_screening.yaml   |  6 +++---
 configs/evaluation/vlm_comparison.yaml         |  8 ++++----
 src/evaluation/background_screening.py         | 19 ++++++++++++++-----
 src/evaluation/config.py                       |  2 ++
 src/vlm/nim.py                                 |  4 +++-
 tests/detection/test_export_predictions_cli.py | 11 ++++++-----
 tests/detection/test_inference_config.py       | 20 +++++++++++---------
 tests/evaluation/test_assistance.py            | 14 ++++++++++++++
 tests/evalu

In [ ]:
!rs-run-assistance-experiment --config configs/evaluation/hrrsd_vlm_comparison.yaml

[job] id=20260922T074655Z_cd71b953
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-run-assistance-experiment/20260922T074655Z_cd71b953
Selected cases: 90; already completed: 0; false negatives excluded from per-box VLM evaluation: 3606
[1/90] hrrsd_external_000002_00005_cmp_0003: completed
[2/90] hrrsd_external_000019_00135_cmp_0002: completed
[3/90] hrrsd_external_000044_00260_cmp_0002: failed
[4/90] hrrsd_external_000057_00360_cmp_0000: completed
[5/90] hrrsd_external_000066_00380_cmp_0004: completed
[6/90] hrrsd_external_000073_00395_cmp_0002: completed
[7/90] hrrsd_external_000078_00409_cmp_0001: completed
[8/90] hrrsd_external_000099_00690_cmp_0003: completed
[9/90] hrrsd_external_000148_00843_cmp_0000: completed
[10/90] hrrsd_external_000148_00843_cmp_0001: completed
[11/90] hrrsd_external_000184_00973_cmp_0003: failed
[12/90] hrrsd_external_000184_00973_cmp_0004: completed
[13/90] hrrsd_external_000184_00973_cmp_0005: faile

In [17]:
%cd /content/retrieval-grounded-remote-sensing

!rs-run-assistance-experiment --config configs/evaluation/vlm_comparison.yaml

/content
[job] id=20260920T212155Z_37ccf116
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-run-assistance-experiment/20260920T212155Z_37ccf116
Selected cases: 82; already completed: 81; false negatives excluded from per-box VLM evaluation: 14
[82/82] nwpu_vhr10_000761_612_cmp_0005: completed
Assistance experiment: /content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/assistance_comparisons/nwpu_yolov8s1024_vlm_40_per_status_seed42_v2
[job] status=succeeded log=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-run-assistance-experiment/20260920T212155Z_37ccf116/run.log


In [ ]:
!rs-run-background-screening --config configs/evaluation/background_screening.yaml

[job] id=20260920T212301Z_d6ce98f7
[job] directory=/content/drive/Othercomputers/My laptop/shared_resources/experiment_outputs/job_logs/rs-run-background-screening/20260920T212301Z_d6ce98f7
[1/27] image_screening::nwpu_vhr10_000001_002: completed
[2/27] image_screening::nwpu_vhr10_000004_005: completed
[3/27] image_screening::nwpu_vhr10_000007_008: completed
[4/27] image_screening::nwpu_vhr10_000013_014: completed
[5/27] image_screening::nwpu_vhr10_000022_023: completed
[6/27] image_screening::nwpu_vhr10_000024_025: completed
[7/27] image_screening::nwpu_vhr10_000025_026: completed
[8/27] image_screening::nwpu_vhr10_000036_037: completed
[9/27] image_screening::nwpu_vhr10_000054_055: completed
[10/27] image_screening::nwpu_vhr10_000067_068: completed
[11/27] image_screening::nwpu_vhr10_000076_077: completed
[12/27] image_screening::nwpu_vhr10_000077_078: completed
[13/27] image_screening::nwpu_vhr10_000082_083: completed
[14/27] image_screening::nwpu_vhr10_000089_090: completed
[15/27]

In [ ]:
import json
from pathlib import Path

experiment_dir = (
    Path("/content/drive/Othercomputers/My laptop/shared_resources")
    / "experiment_outputs"
    / "assistance_comparisons"
    / "nwpu_yolov8s1024_vlm_30_per_status_seed42_v2"
)

summary = json.loads(
    (experiment_dir / "summary.json").read_text(encoding="utf-8")
)

summary

{'selected_cases': 82,
 'completed_cases': 82,
 'failed_cases': 0,
 'excluded_false_negatives': 14,
 'status_counts': {'false_positive': 40,
  'true_positive': 40,
  'class_error': 2}}

In [8]:
import pandas as pd

metrics = pd.read_csv(experiment_dir / "metrics.csv")
display(metrics)

,variant,metric,value,numerator,denominator
0,detector_only,verification_accuracy,0.487805,40.000000,82
1,detector_only,false_accept_rate,1.000000,42.000000,42
2,detector_retrieval,top1_class_accuracy,0.952381,40.000000,42
3,detector_retrieval,precision_at_k,0.942857,39.600000,42
4,detector_retrieval,hit_at_k,0.952381,40.000000,42
5,detector_retrieval,mean_latency_seconds,0.206515,16.934191,82
6,query_only_vlm,verification_accuracy,0.658537,54.000000,82
7,query_only_vlm,macro_decision_f1,0.419753,NaN,82
8,query_only_vlm,class_correction_accuracy,0.000000,0.000000,2
9,query_only_vlm,false_accept_rate,0.666667,28.000000,42


In [10]:
import pandas as pd

records = pd.read_json(
    experiment_dir / "case_results.jsonl",
    lines=True,
)

latest_records = records.drop_duplicates(
    subset="case_id",
    keep="last",
)

display(
    latest_records[
        [
            "case_id",
            "ground_truth_status",
            "run_status",
        ]
    ]
)

,case_id,ground_truth_status,run_status
0,nwpu_vhr10_000040_041_cmp_0000,false_positive,completed
1,nwpu_vhr10_000168_019_cmp_0003,true_positive,completed
2,nwpu_vhr10_000175_026_cmp_0007,false_positive,completed
3,nwpu_vhr10_000198_049_cmp_0006,true_positive,completed
5,nwpu_vhr10_000202_053_cmp_0014,true_positive,completed
...,...,...,...
83,nwpu_vhr10_000307_158_cmp_0004,false_positive,completed
84,nwpu_vhr10_000493_344_cmp_0001,true_positive,completed
85,nwpu_vhr10_000585_436_cmp_0016,false_positive,completed
86,nwpu_vhr10_000632_483_cmp_0004,false_positive,completed


In [12]:
import pandas as pd

metrics = pd.read_csv(experiment_dir / "metrics.csv")

comparison = metrics.pivot(
    index="metric",
    columns="variant",
    values="value",
)

display(comparison.round(4))

variant,configured_policy,detector_only,detector_retrieval,query_only_vlm,retrieval_grounded_vlm
metric,,,,,
auto_accept_coverage,0.3049,NaN,NaN,NaN,NaN
auto_accept_precision,0.8800,NaN,NaN,NaN,NaN
class_correction_accuracy,NaN,NaN,NaN,0.0000,0.0000
decision_accuracy,0.7439,NaN,NaN,NaN,NaN
false_accept_rate,NaN,1.0000,NaN,0.6667,0.7619
hit_at_k,NaN,NaN,0.9524,NaN,NaN
macro_decision_f1,NaN,NaN,NaN,0.4198,0.3370
mean_latency_seconds,NaN,NaN,0.2065,18.0731,20.7545
precision_at_k,NaN,NaN,0.9429,NaN,NaN


## Launch the annotation assistant (optional)

Run this cell only when you want to use the Streamlit app. It securely prompts for the NVIDIA API key when `NIM_API_KEY` is not already set, which also works when the Colab runtime is used through VS Code. The key remains only in the active runtime. The cell starts a temporary public Cloudflare URL; stop it with the following cell when finished.

In [2]:
from getpass import getpass
import re
import time

nim_api_key = os.environ.get("NIM_API_KEY", "").strip()
if not nim_api_key:
    nim_api_key = getpass("NVIDIA NIM API key: ").strip()
if not nim_api_key.startswith("nvapi-"):
    raise ValueError("The supplied NIM_API_KEY is not a valid NVIDIA API key")
os.environ["NIM_API_KEY"] = nim_api_key

checkpoint = SHARED_ROOT / "checkpoints" / "best.pt"
databases = list((SHARED_ROOT / "embeddings").glob("*/lancedb"))
if not checkpoint.is_file():
    raise FileNotFoundError(f"Detector checkpoint not found: {checkpoint}")
if not databases:
    raise FileNotFoundError("No persisted LanceDB artifact was found")

cloudflared = Path("/content/cloudflared")
cloudflared_download = Path("/content/cloudflared.download")
cloudflared_url = (
    "https://github.com/cloudflare/cloudflared/releases/latest/"
    "download/cloudflared-linux-amd64"
)

def cloudflared_is_valid(path):
    if not path.is_file():
        return False
    try:
        result = subprocess.run(
            [str(path), "--version"],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=15,
        )
    except (OSError, subprocess.SubprocessError):
        return False
    return "cloudflared version" in result.stdout.lower()

if not cloudflared_is_valid(cloudflared):
    cloudflared.unlink(missing_ok=True)
    cloudflared_download.unlink(missing_ok=True)
    print("Downloading cloudflared (automatic retries enabled)...")
    subprocess.run(
        [
            "curl",
            "--fail",
            "--location",
            "--retry",
            "5",
            "--retry-all-errors",
            "--connect-timeout",
            "30",
            "--output",
            str(cloudflared_download),
            cloudflared_url,
        ],
        check=True,
    )
    if cloudflared_download.stat().st_size < 10_000_000:
        raise RuntimeError("Downloaded cloudflared file is unexpectedly small")
    cloudflared_download.chmod(0o755)
    if not cloudflared_is_valid(cloudflared_download):
        raise RuntimeError("Downloaded cloudflared executable failed validation")
    cloudflared_download.replace(cloudflared)

for process_name in ("tunnel_process", "streamlit_process"):
    previous = globals().get(process_name)
    if previous is not None and previous.poll() is None:
        previous.terminate()

streamlit_log_path = Path("/content/streamlit.log")
tunnel_log_path = Path("/content/cloudflared.log")
streamlit_log_handle = streamlit_log_path.open("w")
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        "app/annotation_assistant.py",
        "--server.headless=true",
        "--server.address=127.0.0.1",
        "--server.port=8501",
        "--", "--config", "configs/annotation/single_image.yaml",
    ],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    stdout=streamlit_log_handle,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
if streamlit_process.poll() is not None:
    raise RuntimeError(streamlit_log_path.read_text(errors="replace"))

tunnel_log_handle = tunnel_log_path.open("w")
tunnel_process = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(1)
    tunnel_output = tunnel_log_path.read_text(errors="replace")
    match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", tunnel_output)
    if match:
        public_url = match.group(0)
        break

if public_url is None:
    raise RuntimeError(tunnel_log_path.read_text(errors="replace"))
print(f"Open the annotation assistant: {public_url}")

Open the annotation assistant: https://earned-folder-monitors-instead.trycloudflare.com


In [3]:
for process_name in ("tunnel_process", "streamlit_process"):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
for handle_name in ("tunnel_log_handle", "streamlit_log_handle"):
    handle = globals().get(handle_name)
    if handle is not None and not handle.closed:
        handle.close()
print("Annotation assistant stopped.")

Annotation assistant stopped.
